In [5]:
import cv2
import numpy as np
import os

In [ ]:
input_path = r"" # Path
output_path = "night_frame"

os.makedirs(output_path, exist_ok=True)

In [7]:

def add_night(image, dark_strength=0.5, blue_strength=0.15):
    """
    dark_strength : ความมืดรวม 0.0-1.0 (แนะนำ 0.4-0.6)
    blue_strength : ความเข้มโทนน้ำเงิน 0.0-0.3
    """

    # 1. ลด saturation ให้สีจืดลงเหมือนมองในที่มืด
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 1] *= 0.6
    hsv[:, :, 2] *= 0.85
    hsv = np.clip(hsv, 0, 255).astype(np.uint8)
    night = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    # 2. ใส่โทนสีน้ำเงิน
    blue_tint = np.full_like(night, (60, 30, 10))  # BGR
    night = cv2.addWeighted(
        night, 1 - blue_strength,
        blue_tint, blue_strength,
        0
    )

    # 3. มืดลงแบบ overlay สีดำ
    black = np.zeros_like(night)
    night = cv2.addWeighted(
        night, 1 - dark_strength,
        black, dark_strength,
        0
    )

    return night

In [8]:
img_count = 0

for file_name in os.listdir(input_path):

    if not file_name.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    image_path = os.path.join(input_path, file_name)
    img = cv2.imread(image_path)

    if img is None:
        print(f"อ่านไม่ได้: {file_name}")
        continue

    night_img = add_night(img, dark_strength=0.5, blue_strength=0.15)

    save_path = os.path.join(
        output_path,
        f"frame4_{img_count:07d}.jpg"
    )

    cv2.imwrite(save_path, night_img)
    img_count += 1

print(f"Done! Saved {img_count} images.")

Done! Saved 100 images.
